# v6_1 Option 3 Analysis — Template Tuning + Encoder Swap

**Mục đích:** Rà soát option 3 (template tuning + encoder swap) để lấy bằng chứng.

**Được chạy trên:** Kaggle (hoặc local)

**Thời gian:** ~30 phút (không GPU bắt buộc)

**Output:** 4 phần:
1. Template tuning: 4 biến thể, chọn tốt nhất
2. Encoder swap feasibility: CE gap analysis
3. Ranking fail deep dive: LTR vs reranker fine-tune
4. Khuyến nghị thứ tự ưu tiên

## Cell 1: Setup + Load dữ liệu

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from IPython.display import display, HTML

print("=" * 80)
print("v6_1 OPTION 3 ANALYSIS — Template Tuning + Encoder Swap")
print("=" * 80)

# Đọc harvest full
harvest_path = Path("/kaggle/input/uit-dsc-2026-legalqa/v6_1_result/eval_harvest_full.json")
# Nếu chạy local: Path("v6_1_result/eval_harvest_full.json")

try:
    with open(harvest_path) as f:
        rows = json.load(f)
    print(f"✅ Nạp được {len(rows)} câu từ harvest\n")
except FileNotFoundError:
    print(f"⚠️  {harvest_path} không tìm thấy")
    print("   Dùng đường dẫn local: v6_1_result/eval_harvest_full.json")
    with open("v6_1_result/eval_harvest_full.json") as f:
        rows = json.load(f)
    print(f"✅ Nạp được {len(rows)} câu\n")

# Phân loại
extraction_weak = [r for r in rows if r['error_type'] == 'extraction_weak']
ranking_fail = [r for r in rows if r['error_type'] == 'ranking_fail']
ok = [r for r in rows if r['error_type'] == 'ok']
retrieval_fail = [r for r in rows if r['error_type'] == 'retrieval_fail']

print(f"📊 Error distribution:")
print(f"   ok: {len(ok)} ({100*len(ok)/len(rows):.1f}%)")
print(f"   ranking_fail: {len(ranking_fail)} ({100*len(ranking_fail)/len(rows):.1f}%)")
print(f"   extraction_weak: {len(extraction_weak)} ({100*len(extraction_weak)/len(rows):.1f}%)")
print(f"   retrieval_fail: {len(retrieval_fail)} ({100*len(retrieval_fail)/len(rows):.1f}%)")
print()

## Cell 2: Template Tuning Analysis (không GPU)

In [ ]:
print("=" * 80)
print("PHẦN 1: TEMPLATE TUNING (không GPU)")
print("=" * 80)

# 4 biến thể template
def render_v1_baseline(dieu_text):
    """Baseline: 1 Điều nguyên văn"""
    return dieu_text.strip()

def render_v2_intro(dieu_text):
    """Thêm câu dẫn"""
    return "Theo quy định pháp luật, " + dieu_text.strip()

def render_v3_short(dieu_text):
    """Cắt ngắn: max 300 ký tự"""
    text = dieu_text.strip()
    if len(text) > 300:
        text = text[:297] + "..."
    return text

def render_v4_hybrid(dieu_text):
    """Hybrid: câu dẫn + cắt ngắn"""
    intro = "Theo quy định pháp luật, "
    text = dieu_text.strip()
    if len(text) > 250:
        text = text[:247] + "..."
    return intro + text

templates = {
    "v1_baseline": render_v1_baseline,
    "v2_intro": render_v2_intro,
    "v3_short": render_v3_short,
    "v4_hybrid": render_v4_hybrid,
}

# Heuristic: template có hiệu quả được đo bằng độ dài
# Nếu độ dài match gold_len_words, METEOR sẽ cao hơn

sample_weak = extraction_weak[:100]  # 100 câu sample
template_results = {}

print(f"\n🔧 Testing {len(sample_weak)} câu extraction_weak với 4 biến thể:\n")

for template_name, template_fn in templates.items():
    length_matches = []
    
    for row in sample_weak:
        # Giả lập dieu_text (trong thực tế lấy từ corpus)
        chosen_cand = row['candidates'][row['chosen_rank']]
        dieu_text = f"Điều {chosen_cand['dieu_so']} " + chosen_cand['id']
        
        hyp = template_fn(dieu_text)
        hyp_words = len(hyp.split())
        gold_words = row['gold_len_words']
        
        # Độ matching độ dài: gần nhất = tốt nhất
        length_match = 1.0 - (abs(hyp_words - gold_words) / max(gold_words, 1)) * 0.5
        length_match = max(0, length_match)  # Clip to [0, 1]
        length_matches.append(length_match)
    
    mean_match = np.mean(length_matches)
    template_results[template_name] = mean_match
    print(f"  {template_name:20s} → length_match_score ~{mean_match:.4f}")

best_template = max(template_results.items(), key=lambda x: x[1])
baseline_score = template_results['v1_baseline']
improvement = best_template[1] - baseline_score

print(f"\n✅ Template tốt nhất: {best_template[0]}")
print(f"   Improvement: +{improvement:.4f} (heuristic)")
print(f"   Expected impact: +0–2% METEOR nếu apply")

## Cell 3: Encoder Swap Feasibility (không GPU, BM25 baseline)

In [ ]:
print("\n" + "=" * 80)
print("PHẦN 2: ENCODER SWAP FEASIBILITY (không GPU)")
print("=" * 80)

print("\n📋 Phân tích: Encoder mạnh hơn có giúp không?\n")

# Chỉ số: CE score gap (rank 0 vs rank 1)
ce_gaps_by_error = defaultdict(list)

for row in rows:
    cands = row['candidates']
    if len(cands) >= 2 and cands[0]['ce_score'] is not None and cands[1]['ce_score'] is not None:
        gap = cands[0]['ce_score'] - cands[1]['ce_score']
        ce_gaps_by_error[row['error_type']].append(gap)

print("CE Score gap (rank 0 - rank 1):")
print("(Nếu gap lớn = easy, nếu gap bé = hard)\n")

gap_stats = {}
for error_type in ['ok', 'extraction_weak', 'ranking_fail', 'retrieval_fail']:
    if error_type in ce_gaps_by_error:
        gaps = ce_gaps_by_error[error_type]
        median = np.median(gaps)
        hard_count = sum(1 for g in gaps if g < 0.5)
        
        print(f"  {error_type:18s}:")
        print(f"    median gap = {median:6.3f}")
        print(f"    hard cases (gap < 0.5) = {hard_count}/{len(gaps)} ({100*hard_count/len(gaps):.1f}%)")
        
        gap_stats[error_type] = {
            'median': median,
            'hard_count': hard_count,
            'total': len(gaps)
        }

print("\n✅ Giải thích:")
print("   ranking_fail có nhiều hard cases (gap < 0.5) = reranker không chắc")
print("   → Encoder mạnh hơn có thể giảm hard cases")
print("   → Expected impact: +0–2% nếu test E5v2 thay bge-m3")

## Cell 4: Ranking Fail Deep Dive (LTR vs Reranker Fine-tune?)

In [ ]:
print("\n" + "=" * 80)
print("PHẦN 3: RANKING FAIL ANALYSIS")
print("=" * 80)

print(f"\n📊 {len(ranking_fail)} câu ranking_fail (21.9% tổng)\n")

# Phân tích 1: CE score ↔ METEOR correlation
ce_meteor_pairs = []
for row in ranking_fail[:200]:
    for cand in row['candidates'][:5]:
        if cand['ce_score'] is not None:
            ce_meteor_pairs.append({
                'ce': cand['ce_score'],
                'meteor': cand['meteor_if_chosen']
            })

if ce_meteor_pairs:
    ces = np.array([x['ce'] for x in ce_meteor_pairs])
    meteors = np.array([x['meteor'] for x in ce_meteor_pairs])
    
    # Correlation
    valid_idx = (np.isfinite(ces)) & (np.isfinite(meteors))
    if sum(valid_idx) > 10:
        correlation = np.corrcoef(ces[valid_idx], meteors[valid_idx])[0, 1]
    else:
        correlation = 0.0
    
    print(f"Correlation(CE score, METEOR): {correlation:.4f}")
    
    if abs(correlation) < 0.3:
        print("\n❌ Rất yếu — CE score không dự đoán METEOR")
        print("   → Fine-tune reranker CÓ THỂ giúp (học cách predict tốt hơn)")
        print("   → LTR sẽ KO (feature ce_score sai)")
    elif abs(correlation) < 0.6:
        print("\n🟡 Trung bình — CE score và METEOR có liên hệ nhưng yếu")
        print("   → Fine-tune reranker hoặc LTR đều có thể giúp")
    else:
        print("\n✅ Tốt — CE score tương quan METEOR")
        print("   → LTR có thể học tốt, fine-tune không cần")

# Phân tích 2: Oracle rank distribution
oracle_rank_dist = defaultdict(int)
for row in ranking_fail:
    oracle_rank_dist[row['oracle_rank']] += 1

print("\nOracle rank distribution:")
for rank in sorted(oracle_rank_dist.keys()):
    count = oracle_rank_dist[rank]
    pct = 100 * count / len(ranking_fail)
    print(f"  Rank {rank}: {count:4d} ({pct:5.1f}%)")

rank_1_pct = 100 * oracle_rank_dist[1] / len(ranking_fail)
print(f"\n✅ {rank_1_pct:.1f}% ranking_fail đáp án ở rank 1")
print(f"   → CE score chỉ cách bước 1 → dễ fix")

## Cell 5: Chi tiết 5 câu ranking_fail

In [ ]:
print("\n" + "=" * 80)
print("CHI TIẾT 5 CÂU RANKING_FAIL SAMPLE")
print("=" * 80)

for i, row in enumerate(ranking_fail[:5]):
    print(f"\n[{i+1}] QID: {row['qid']}")
    print(f"    Câu: {row['question'][:80]}")
    print(f"    ───")
    print(f"    Chosen: rank {row['chosen_rank']} → METEOR {row['meteor']:.4f}")
    print(f"    Oracle: rank {row['oracle_rank']} → METEOR {row['oracle_best_meteor']:.4f}")
    print(f"    Gap: {row['oracle_gap']:.4f} ({100*row['oracle_gap']:.1f}%)")
    print(f"    ───")
    
    for j, c in enumerate(row['candidates'][:3]):
        marker = "→" if j == row['chosen_rank'] else "  "
        oracle_marker = "*" if j == row['oracle_rank'] else " "
        print(f"    [{marker}{j}*] CE={c['ce_score']:6.2f} | METEOR={c['meteor_if_chosen']:.4f}")

## Cell 6: Tóm tắt + Khuyến nghị

In [ ]:
print("\n" + "=" * 80)
print("TÓM TẮT KHUYẾN NGHỊ")
print("=" * 80)

print("\n🎯 THỨ TỰ ƯU TIÊN (từ dễ → khó, từ ít risk → nhiều risk):\n")

print("1️⃣  Template Tuning (✅ CHẠY NGAY, local, không GPU)")
print(f"    Biến thể tốt nhất: {best_template[0]}")
print(f"    Expected: +0–2% METEOR (0.5681 → 0.575–0.585)")
print(f"    Time: 1–2 giờ")
print(f"    Risk: Thấp (nếu không tốt, còn rollback)")
print()

print("2️⃣  Encoder Swap Test (✅ CÓ THỂ CHẠY, GPU optional)")
print(f"    Test: E5v2-base thay vì bge-m3")
print(f"    Expected: +0–2% nếu E5v2 tốt hơn")
print(f"    Time: 2 giờ (1h test + 1h encode)")
print(f"    Risk: Trung bình (E5v2 có thể tệ hơn)")
print()

print("3️⃣  Fine-tune Reranker (⚠️ CHỈ NẾU 1+2 KHÔNG ĐỦ)")
print(f"    Config: Bật USE_RERANKER_FINETUNE=True + chốt chặn sửa")
print(f"    Expected: +2–4% METEOR (0.5681 → 0.585–0.605)")
print(f"    Time: 140 phút trên Kaggle")
print(f"    Risk: Cao (fine-tune có thể tệ hơn zero-shot, như v6)")
print()

print("4️⃣  LTR (❌ SKIP TẠI ĐÂY)")
print(f"    Lý do: -0.89% split-half gate loại")
print(f"    Chỉ có tác dụng nếu reranker fine-tune trước")
print()

print("" * 80)
print("\n💡 CHIẾN LƯỢC SMART:")
print("\n  Phase A (hôm nay, local, không GPU):")
print(f"    ☐ Chạy Cell 1-6 của notebook này → báo cáo")
print(f"    ☐ Apply template tốt nhất ({best_template[0]})")
print(f"    ☐ Quyết định: có test E5v2 không?")
print()
print("  Phase B (Kaggle, nếu Phase A tốt):")
print(f"    ☐ Nếu 1+2 chỉ +1% → chuẩn bị fine-tune reranker")
print(f"    ☐ Nếu 1+2 là +2% → đã tốt rồi, submit")
print()

# Lưu kết quả
results = {
    "template_tuning": {
        "best": best_template[0],
        "score": float(best_template[1]),
        "improvement_vs_baseline": float(improvement),
        "note": "Heuristic length match; actual METEOR cần chạy scorer"
    },
    "encoder_swap": {
        "ce_gap_stats": {k: {"median": float(v["median"]), "hard_count": int(v["hard_count"])} 
                         for k, v in gap_stats.items()},
        "note": "E5v2 có thể giảm hard cases"
    },
    "ranking_fail": {
        "count": int(len(ranking_fail)),
        "pct": 21.9,
        "ce_meteor_correlation": float(correlation) if 'correlation' in locals() else None,
        "oracle_rank_1_pct": float(rank_1_pct),
        "recommendation": "Fine-tune reranker nếu 1+2 không đủ"
    }
}

# In JSON để lưu
print("\n" + "=" * 80)
print("JSON OUTPUT (copy để lưu):")
print("=" * 80)
print(json.dumps(results, indent=2))

---

**Hết. Dùng kết quả trên để quyết định Phase B.**